In [ ]:
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle


In [ ]:
# 2. LOAD DATASET
df = pd.read_csv("/content/icecream_sales_linear.csv")

In [ ]:
# View first 5 rows
df.head()

,temperature,humidity,sales
0,23.11,68.52,112.23
1,43.28,35.05,216.91
2,35.62,39.70,185.35
3,30.95,83.91,119.06
4,15.46,66.39,69.71


In [8]:
# Shape of dataset
df.shape
# Data types & missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   temperature  200 non-null    float64
 1   humidity     200 non-null    float64
 2   sales        200 non-null    float64
dtypes: float64(3)
memory usage: 4.8 KB


In [ ]:
# Descriptive statistics
df.describe()

,temperature,humidity,sales
count,200.000000,200.000000,200.000000
mean,26.940150,60.262800,122.979250
std,10.321409,17.579967,50.065606
min,10.190000,30.300000,29.760000
25%,18.002500,45.685000,79.077500
50%,27.305000,62.500000,123.020000
75%,36.487500,74.535000,165.200000
max,44.540000,89.430000,224.400000


In [ ]:
df.mean(numeric_only=True)
df.median(numeric_only=True)
df.mode().iloc[0]

,0
temperature,15.46
humidity,71.76
sales,108.61


In [ ]:
# 5. FEATURES & TARGET
X = df[['temperature', 'humidity']]   # 2 features
y = df['sales']                        # target

In [ ]:
# 6. TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# 7. LINEAR REGRESSION
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

LinearRegression()

In [ ]:
# MODEL PREDICTIONS
# Predictions on training data
y_train_pred = lr_model.predict(X_train)

# Predictions on testing data (unseen data)
y_test_pred = lr_model.predict(X_test)

In [ ]:
# EVALUATION FUNCTION

def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2


In [ ]:
# TESTING PERFORMANCE

test_mae, test_mse, test_rmse, test_r2 = evaluate_model(
    y_test, y_test_pred
)

print("\nTesting Set Performance:")
print("MAE :", test_mae)
print("MSE :", test_mse)
print("RMSE:", test_rmse)
print("R2  :", test_r2)



Testing Set Performance:
MAE : 9.886188107004186
MSE : 147.7994071867387
RMSE: 12.157277951364717
R2  : 0.94407490474947


In [ ]:
# 9. POLYNOMIAL REGRESSION

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Create Polynomial Regression pipeline
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr', LinearRegression())
])

# MODEL TRAINING (POLYNOMIAL)
poly_pipeline.fit(X_train, y_train)


Pipeline(steps=[('poly', PolynomialFeatures(include_bias=False)),
                ('lr', LinearRegression())])

In [33]:
# Predictions on training data
y_poly_train_pred = poly_pipeline.predict(X_train)

# Predictions on testing data
y_poly_test_pred = poly_pipeline.predict(X_test)


In [34]:
# Evaluate training performance
poly_train_mae, poly_train_mse, poly_train_rmse, poly_train_r2 = evaluate_model(
    y_train, y_poly_train_pred
)

print("Polynomial Regression - Training Set Performance:")
print("MAE :", poly_train_mae)
print("MSE :", poly_train_mse)
print("RMSE:", poly_train_rmse)
print("R2  :", poly_train_r2)


Polynomial Regression - Training Set Performance:
MAE : 7.348325913038283
MSE : 82.46350667755432
RMSE: 9.080941948804337
R2  : 0.966430506416647


In [ ]:
# 11. GRID SEARCH CV

param_grid = {
    'poly__degree': [1, 2, 3, 4]
}

grid = GridSearchCV(
    poly_pipeline,
    param_grid,
    cv=5,
    scoring='r2'
)

grid.fit(X_train, y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('poly',
                                        PolynomialFeatures(include_bias=False)),
                                       ('lr', LinearRegression())]),
             param_grid={'poly__degree': [1, 2, 3, 4]}, scoring='r2')

In [39]:
# Best parameters
grid.best_params_


{'poly__degree': 1}

In [ ]:
# Best model evaluation
best_model = grid.best_estimator_
y_best_pred = best_model.predict(X_test)

best_metrics = evaluate_model(y_test, y_best_pred)

# Print in the desired format
print("Testing Set Performance:")
print(f"MAE : {best_metrics[0]}")
print(f"MSE : {best_metrics[1]}")
print(f"RMSE: {best_metrics[2]}")
print(f"R2  : {best_metrics[3]}")

best_metrics

Testing Set Performance:
MAE : 9.886188107004184
MSE : 147.79940718673868
RMSE: 12.157277951364716
R2  : 0.9440749047494701


(9.886188107004184,
 147.79940718673868,
 np.float64(12.157277951364716),
 0.9440749047494701)

In [ ]:
# 12. ELASTIC NET
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)
elastic.fit(X_train, y_train)

y_elastic_pred = elastic.predict(X_test)

elastic_metrics = evaluate_model(y_test, y_elastic_pred)  # Will print automatically
# elastic_metrics will still contain the tuple


# 12. ELASTIC NET
elastic = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)
elastic.fit(X_train, y_train)

y_elastic_pred = elastic.predict(X_test)

elastic_metrics = evaluate_model(y_test, y_elastic_pred)

# Print in the format you want
print("Testing Set Performance:")
print(f"MAE : {elastic_metrics[0]}")
print(f"MSE : {elastic_metrics[1]}")
print(f"RMSE: {elastic_metrics[2]}")
print(f"R2  : {elastic_metrics[3]}")

# elastic_metrics still contains the tuple for further use

Testing Set Performance:
MAE : 9.886827232002068
MSE : 147.8774853067621
RMSE: 12.160488695227759
R2  : 0.9440453611512757


In [ ]:
# 13. SAVE MODEL

with open("best_sales_model.pkl", "wb") as file:
    pickle.dump(best_model, file)


In [49]:
from google.colab import files

# Download the .pkl file
files.download('best_sales_model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>